In [1]:
import os
import torch
import random
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer

ModuleNotFoundError: No module named 'torch'

In [ ]:
class DataManager:
    def __init__(self, url= None):
        self.url = url
        self.max_seq_len = None       # store the max sequence length
        self.num_sentences = None     # store number of sentences
        self.texts = None             # store all sentences
        self.labels = None            # store all labels
        self.num_seqs = None         # store sequences of indices
        self.vocab_size = None


    def read_data(self, file_path):
        df = pd.read_csv(file_path, encoding = "ISO-8859-1")
        df['label'] = df['v1'].apply(lambda x: 1 if x == 'spam' else 0)
        labels, texts = df['label'].to_numpy(), df['v2'].tolist()
        self.texts= texts
        self.labels = torch.from_numpy(labels)

    def transform_to_numbers(self):
        self.num_seqs = self.tokenizer(self.texts, return_tensors='pt', truncation=True, padding=True)['input_ids']
        self.num_sentences, self.max_seq_len = self.num_seqs.shape

    def build_vocabulary(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.word2idx = {w: i for w,i in self.tokenizer.vocab.items()}
        self.idx2word = {i:w for w,i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)
        self.min_index = min(self.word2idx.values())
        self.max_index = max(self.word2idx.values())

    def process_data(self):
        self.build_vocabulary()
        self.transform_to_numbers()


    def train_valid_test_split(self, train_ratio= 0.8, test_ratio=0.1):
        train_size = int(self.num_sentences*train_ratio) +1
        test_size = int(self.num_sentences*test_ratio) +1
        valid_size = self.num_sentences - (train_size + test_size)
        data_indices = list(range(self.num_sentences))
        random.shuffle(data_indices)
        train_set_data = self.num_seqs[data_indices[:train_size]]
        train_set_labels = self.labels[data_indices[:train_size]]
        train_set = torch.utils.data.TensorDataset(train_set_data, train_set_labels)
        test_set_data = self.num_seqs[data_indices[-test_size:]]
        test_set_labels = self.labels[data_indices[-test_size:]]
        test_set = torch.utils.data.TensorDataset(test_set_data, test_set_labels)
        valid_set_data = self.num_seqs[data_indices[train_size:-test_size]]
        valid_set_labels = self.labels[data_indices[train_size:-test_size]]
        valid_set = torch.utils.data.TensorDataset(valid_set_data, valid_set_labels)
        self.train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
        self.test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
        self.valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False)

In [ ]:
dm = DataManager()
dm.read_data("./data.csv")
dm.process_data()
dm.train_valid_test_split()

In [ ]:
import gensim.downloader as api

In [ ]:
class NewsModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, embed_matrix=None):
    super(NewsModel, self).__init__()

    if embed_matrix is not None:
      self.embedding_layer = nn.Embedding.from_pretrained(embed_matrix)
    else:
      self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)
    self.rnn_layer = nn.GRU(embedding_dim, hidden_dim, num_layers=1, batch_first=True)
    self.dense_layer = nn.Linear(hidden_dim, 2)

  def forward(self, x):
    e = self.embedding_layer(x)
    h,_ = self.rnn_layer(e)
    h = h[:, -1, :]
    y = self.dense_layer(h)
    return y

In [ ]:
def train_epoch(model, optimizer, loader, criterion, device):
  model.train()
  losses = 0
  accuracies = 0
  for x, y in loader:
    x, y = x.to(device), y.to(device)
    pred = model(x)
    ypred = pred.argmax(-1)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    acc = (y == ypred).sum(0) / len(y)
    accuracies += acc.item()
    losses += loss.item()
  return losses / len(loader), accuracies / len(loader)

def test_epoch(model, loader, criterion, device):
    model.eval()
    losses = 0
    accuracies = 0
    with torch.no_grad():
      for x,y in loader:
        x,y = x.to(device), y.to(device)
        pred = model(x)
        ypred = pred.argmax(-1)
        loss = criterion(pred, y)
        acc = (y == ypred).sum(0) / len(y)
        accuracies += acc.item()
        losses += loss.item()
    return losses / len(loader), accuracies / len(loader)


In [ ]:
class RNN_NewsModel:
    def __init__(self, run_mode="scratch", embed_model="glove-wiki-gigaword-300", embed_size=128, hidden_size=128, data_manager=None):
        self.embed_path = "embeddings/E.npy"
        self.embed_model = embed_model
        self.embed_size = embed_size
        self.run_mode = run_mode
        if run_mode != 'scratch':
            self.embed_size = int(self.embed_model.split("-")[-1])
        self.data_manager = data_manager
        self.vocab_size = self.data_manager.vocab_size
        self.word2idx = self.data_manager.word2idx
        self.embed_matrix = np.zeros((self.vocab_size, self.embed_size))
        self.run_mode = run_mode
        self.hidden_size = hidden_size
        self.model = None

    def build_embedding_matrix(self):
        if os.path.exists(self.embed_path): # file existed
            self.embed_matrix = np.load(self.embed_path) # Load the file for embedding matrix if existed
        else: # file not existed or first-time run
            self.word2vect = api.load(self.embed_model) # load embedding model
            for word, idx in self.word2idx.items():
                try:
                    self.embed_matrix[idx] = self.word2vect.word_vec(word) # assign weight for the corresponding word and index
                except KeyError: # word cannot be found
                    pass
            np.save(self.embed_path, self.embed_matrix)

    def build(self):

      if self.run_mode == 'scratch':
        embed_matrix = None
      else: # init-fine-tune
        self.build_embedding_matrix()
        embed_matrix = torch.from_numpy(self.embed_matrix)
        embed_matrix.requires_grad = True

      model = SpamDetectionModel(self.vocab_size, self.embed_size, self.hidden_size, embed_matrix)
      self.criterion = nn.CrossEntropyLoss(reduction="mean")
      return model


    def train(self, model, device, num_epochs):
      optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
      for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_epoch(model, optimizer, self.data_manager.train_loader,
                                            self.criterion, device)
        val_loss, val_acc = test_epoch(model, self.data_manager.valid_loader,
                                       self.criterion, device)
        msg = f"Epoch: {epoch}/{num_epochs} - train loss = {train_loss:.3f} - train accuracy = {train_acc*100:.3f}%"
        msg =  msg + f"- val loss = {val_loss:.3f} - val accuracy = {val_acc*100:.3f}%"
        print(msg)

    def evaluate(self, model, device):
      loss, acc = test_epoch(model, self.data_manager.test_loader, self.criterion, device)
      print(f"Test loss = {loss:.3f} - Test accuracy = {acc*100:.3f}%")


In [ ]:
rnn1 = RNN_NewsModel(data_manager=dm, run_mode="scratch")
rnn2 = RNN_NewsModel(data_manager=dm, run_mode="init-fine-tune")

In [ ]:
model = rnn1.build()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
rnn1.train(model, device, num_epochs=20)

In [ ]:
rnn1.evaluate(model, device)